# 🔍 Microsoft Learn Documentation Chatbot

An interactive chatbot powered by **Azure AI Agent Framework** that provides answers about Microsoft technologies using the official **Microsoft Learn MCP** (Model Context Protocol) server.

## Features
- 💬 Real-time conversational interface with Gradio
- 📚 Access to official Microsoft documentation via MCP
- 🔄 Multi-turn conversations (agent remembers context)
- 💾 Persistent conversation history (saved to local storage)

## Architecture
<img src="architecture.jpg">

<img src="gradio.jpg">

## 1. Setup & Configuration

In [1]:
# %pip install agent-framework azure-identity

In [2]:
import asyncio
import gradio as gr
import json
import os
import sys
import uuid

from agent_framework import AgentProtocol, AgentRunResponse, ChatMessage, HostedMCPTool
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import DefaultAzureCredential
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path
from typing import Any, Optional

In [3]:
print(f"Python version: {sys.version}")

Python version: 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]


In [4]:
load_dotenv("azure.env")

AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AZURE_AI_MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

print(f"Model: {AZURE_AI_MODEL_DEPLOYMENT_NAME}")

Model: gpt-4.1


In [5]:
MCP_URL = "https://learn.microsoft.com/api/mcp"

print(f"MCP URL to use: {MCP_URL}")

MCP URL to use: https://learn.microsoft.com/api/mcp


## 2. Conversation History Manager

This class handles saving and loading conversation history to/from a JSON file.

In [6]:
class ConversationHistoryManager:
    """
    Manages persistent conversation history storage.
    """
    def __init__(self, storage_path: str = "chat_history.json"):
        self.storage_path = Path(storage_path)
        self.data = self._load_data()

    def _load_data(self) -> dict:
        """Load existing data from storage or create new structure."""
        if self.storage_path.exists():
            try:
                with open(self.storage_path, "r", encoding="utf-8") as f:
                    return json.load(f)
            except (json.JSONDecodeError, IOError) as e:
                print(f"❌ Error loading history: {e}. Starting fresh.")

        return {"conversations": {}, "current_conversation_id": None}

    def _save_data(self) -> None:
        """Save current data to storage."""
        try:
            with open(self.storage_path, "w", encoding="utf-8") as f:
                json.dump(self.data, f, indent=2, ensure_ascii=False)
        except IOError as e:
            print(f"❌ Error saving history: {e}")

    def create_conversation(self) -> str:
        """Create a new conversation and set it as current."""
        conv_id = str(uuid.uuid4())[:8]
        now = datetime.now().isoformat()

        self.data["conversations"][conv_id] = {
            "id": conv_id,
            "title": "New Conversation",
            "created_at": now,
            "updated_at": now,
            "messages": []
        }
        self.data["current_conversation_id"] = conv_id
        self._save_data()

        return conv_id

    def get_current_conversation_id(self) -> Optional[str]:
        """Get the current conversation ID, creating one if needed."""
        if not self.data["current_conversation_id"]:
            self.create_conversation()
        return self.data["current_conversation_id"]

    def set_current_conversation(self, conv_id: str) -> bool:
        """Set the current conversation by ID."""
        if conv_id in self.data["conversations"]:
            self.data["current_conversation_id"] = conv_id
            self._save_data()
            return True
        return False

    def get_messages(self, conv_id: Optional[str] = None) -> list:
        """Get messages for a conversation."""
        conv_id = conv_id or self.get_current_conversation_id()
        if conv_id and conv_id in self.data["conversations"]:
            return self.data["conversations"][conv_id]["messages"]
        return []

    def add_message(self,
                    role: str,
                    content: str,
                    conv_id: Optional[str] = None) -> None:
        """Add a message to a conversation."""
        conv_id = conv_id or self.get_current_conversation_id()
        if not conv_id:
            conv_id = self.create_conversation()

        conversation = self.data["conversations"][conv_id]
        conversation["messages"].append({
            "role": role,
            "content": content,
            "timestamp": datetime.now().isoformat()
        })
        conversation["updated_at"] = datetime.now().isoformat()

        # Update title from first user message
        if role == "user" and conversation["title"] == "New Conversation":
            conversation["title"] = content[:50] + ("..." if len(content) > 50
                                                    else "")

        self._save_data()

    def get_conversation_list(self) -> list:
        """Get a list of all conversations with metadata."""
        conversations = []
        for conv_id, conv_data in self.data["conversations"].items():
            conversations.append({
                "id": conv_id,
                "title": conv_data["title"],
                "created_at": conv_data["created_at"],
                "updated_at": conv_data["updated_at"],
                "message_count": len(conv_data["messages"])
            })

        conversations.sort(key=lambda x: x["updated_at"], reverse=True)
        return conversations

    def delete_conversation(self, conv_id: str) -> bool:
        """Delete a conversation."""
        if conv_id in self.data["conversations"]:
            del self.data["conversations"][conv_id]

            if self.data["current_conversation_id"] == conv_id:
                self.data["current_conversation_id"] = None

            self._save_data()
            return True
        return False

    def export_conversation(self, conv_id: Optional[str] = None) -> str:
        """Export a conversation as formatted text."""
        conv_id = conv_id or self.get_current_conversation_id()
        if not conv_id or conv_id not in self.data["conversations"]:
            return "No conversation found."

        conv = self.data["conversations"][conv_id]
        lines = [
            f"# {conv['title']}", f"Created: {conv['created_at']}",
            f"Last updated: {conv['updated_at']}", "", "---", ""
        ]

        for msg in conv["messages"]:
            role_icon = "👤" if msg["role"] == "user" else "🤖"
            lines.append(f"{role_icon} **{msg['role'].title()}:**")
            lines.append(msg["content"])
            lines.append("")

        return "\n".join(lines)

In [7]:
# Initialize the history manager
history_manager = ConversationHistoryManager("history.json")

# Show existing conversations
conversations = history_manager.get_conversation_list()
print(f"📚 Found {len(conversations)} existing conversation(s)")

for conv in conversations[:10]:
    print(
        f"   • [{conv['id']}] {conv['title']} ({conv['message_count']} messages)"
    )

📚 Found 1 existing conversation(s)
   • [e0763025] How to do OCR? (4 messages)


## 3. Agent Configuration

The agent uses the **Microsoft Learn MCP** endpoint and maintains conversation context for follow-up questions.

In [8]:
# Maximum number of previous messages to include for context
MAX_CONTEXT_MESSAGES = 10

In [9]:
print(f"MCP URL is: {MCP_URL}")

MCP URL is: https://learn.microsoft.com/api/mcp


In [10]:
# MCP Configuration
MCP_CONFIG = {
    "name": "Microsoft Learn MCP",
    "url": MCP_URL,
    "approval_mode": "never_require",
}

# Agent system prompt - updated to handle follow-up questions
SYSTEM_INSTRUCTIONS = """
You are a knowledgeable Microsoft Azure documentation assistant. 
Your role is to help users find information about Microsoft Azure technologies, 
Azure services, and development tools using the official Microsoft Learn documentation.

Guidelines:
- Provide accurate, up-to-date information from Microsoft Learn,
- Include relevant documentation links when available,
- Format code examples with proper syntax highlighting,
- Be concise but comprehensive in your explanations,
- If you're unsure about something, say so and suggest where to find more information,
- Always cite your sources with links to the official documentation at the end of the answer.

IMPORTANT - Multi-turn conversation support:
- You have access to the conversation history,
- When the user asks follow-up questions like "translate this", "summarize that", 
  "explain in more detail", "give me an example", refer to your previous responses,
- You can translate, summarize, expand, or modify your previous answers based on user requests,
- If the user asks to translate, translate your last response to the requested language.
- Always answer in English.
""".strip()

## 4. Core Agent Functions with Conversation Context

In [11]:
def build_conversation_context(history: list,
                               max_messages: int = MAX_CONTEXT_MESSAGES
                               ) -> str:
    """
    Build a conversation context string from chat history.
    
    Parameters
    ----------
    history : list
        List of message dictionaries with 'role' and 'content' keys.
    max_messages : int
        Maximum number of recent messages to include.
        
    Returns
    -------
    str
        Formatted conversation context string.
    """
    if not history:
        return ""

    # Take only the most recent messages
    recent_history = history[-max_messages:]

    context_parts = ["\nCONVERSATION HISTORY"]

    for msg in recent_history:
        role = msg.get("role", "unknown").upper()
        content = msg.get("content", "")
        context_parts.append(f"\n[{role}]: {content}")

    context_parts.append("\nEND OF HISTORY")
    context_parts.append(
        "\nPlease respond to the user's latest message, taking into account the conversation history above."
    )

    return "\n".join(context_parts)

In [12]:
async def execute_agent_query(query: str, agent: AgentProtocol) -> str:
    """
    Execute a query against the MCP-enabled agent.
    """
    result = await agent.run(query, store=False)

    # Handle any approval requests automatically
    while len(result.user_input_requests) > 0:
        new_inputs: list[Any] = [query]
        for user_input in result.user_input_requests:
            new_inputs.append(
                ChatMessage(role="assistant", contents=[user_input]))
            new_inputs.append(
                ChatMessage(
                    role="user",
                    contents=[user_input.create_response(approved=True)]))
        result = await agent.run(new_inputs, store=False)

    return str(result)

In [13]:
async def get_agent_response(message: str,
                             conversation_history: list = None) -> str:
    """
    Process a user message with conversation context and return the agent's response.

    Parameters
    ----------
    message : str
        The current user message.
    conversation_history : list, optional
        Previous messages in the conversation for context.

    Returns
    -------
    str
        The agent's response.
    """
    # Build the full query with conversation context
    if conversation_history:
        context = build_conversation_context(conversation_history)
        full_query = f"{context}\n\n[USER's NEW MESSAGE]: {message}"
    else:
        full_query = message

    async with DefaultAzureCredential() as credential:
        async with AzureAIAgentClient(credential=credential).create_agent(
                name="MicrosoftLearnAssistant",
                instructions=SYSTEM_INSTRUCTIONS,
                tools=HostedMCPTool(**MCP_CONFIG),
        ) as agent:
            return await execute_agent_query(full_query, agent)

In [14]:
def run_async_in_thread(coro):
    """
    Run an async coroutine in a new thread with its own event loop.
    """
    def run_in_new_loop():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(coro)
        finally:
            loop.close()

    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(run_in_new_loop)
        return future.result()


def chat_sync(message: str, conversation_history: list = None) -> str:
    """
    Synchronous wrapper for the async chat function with conversation context.
    
    Parameters
    ----------
    message : str
        User message.
    conversation_history : list, optional
        Previous messages for context.

    Returns
    -------
    str
        Agent response.
    """
    return run_async_in_thread(
        get_agent_response(message, conversation_history))

## 5. Test Multi-Turn Conversation

Let's test the agent with a follow-up question.

In [15]:
# First question
question1 = "How to do object detection wtih Azure ML?"
print(f"\n👤 User: {question1}")

response1 = chat_sync(question1)
print("\033[1;31;34m")
print(f"🤖 Agent: {response1}")

# Build history
history = [{
    "role": "user",
    "content": question1
}, {
    "role": "assistant",
    "content": response1
}]


👤 User: How to do object detection wtih Azure ML?

🤖 Agent: To do object detection with Azure Machine Learning (Azure ML), you can use Azure ML's automated machine learning (AutoML) features for computer vision. Here's a high-level overview of the process:

## Steps to Train an Object Detection Model with Azure ML

1. **Prepare Data:**
   - Organize your labeled images in a format supported by Azure ML (e.g., Pascal VOC, COCO, or JSONL).
   - Upload your images to an Azure storage blob and register them as a data asset in Azure ML.

2. **Set Up Your Environment:**
   - Install the latest Azure ML SDK (`azure-ai-ml` version 2).
   - Make sure you have a compute cluster with a GPU (recommended).

3. **Create an MLTable:**
   - Convert your annotations to the JSONL format and register an MLTable data asset for your images and labels.

4. **Configure the AutoML Job:**
   - Define the task as `image_object_detection`.
   - Set parameters like `primary_metric` (e.g., `mean_average_precision

In [16]:
# Follow-up question
question2 = "Summarize the answer in French"
print(f"\n👤 User: {question2}")

response2 = chat_sync(question2, history)
print("\033[1;31;34m")
print(f"🤖 Agent: {response2}")


👤 User: Summarize the answer in French

🤖 Agent: Here’s a summary of the previous answer in French:

Pour faire de la détection d’objets avec Azure Machine Learning (Azure ML), il faut :

1. Préparer les données : organiser et annoter les images, puis les déposer dans un stockage Azure et les enregistrer en tant qu’actif de données.
2. Configurer l’environnement : installer le SDK Azure ML et utiliser un cluster de calcul avec GPU.
3. Créer un MLTable avec les annotations au format JSONL.
4. Configurer un job AutoML pour la tâche image_object_detection en définissant les paramètres principaux (ex. : « mean_average_precision »).
5. (Optionnel) Optimiser via une recherche d’hyperparamètres.
6. Soumettre le job pour entraîner le modèle.
7. Déployer le modèle entraîné comme endpoint web.
8. Visualiser les résultats en affichant les prédictions sur les images.

Pour plus de détails : [Tutoriel officiel Azure ML - Détection d’objets](https://learn.microsoft.com/en-us/azure/machine-learning/

In [17]:
# Follow-up question
question3 = "What is the pricing model?"
print(f"\n👤 User: {question3}")

response3 = chat_sync(question3, history)
print("\033[1;31;34m")
print(f"🤖 Agent: {response3}")


👤 User: What is the pricing model?

🤖 Agent: The pricing model for Azure Machine Learning (Azure ML)—including training object detection models—depends on several key factors:

## 1. Compute Resources
- **You pay for the compute instances/clusters (VMs) that you use** during training, validation, model tuning, and deployment.
- Pricing is based on the size/type of the VM (e.g., GPU vs. CPU, Standard_NC6s_v3, Standard_D2_v2, etc.).
- You are charged by the **hour**, and only for the time the compute is running (provisioned and in use).
- More powerful GPUs (recommended for deep learning and object detection) are more expensive.

## 2. Storage
- **Data storage** in Azure Blob Storage (for images and data assets) is billed based on the type of storage and the volume used.
- Model artifacts (trained models, logs, outputs) are also stored and billed.

## 3. Azure ML Service Charges
- There is **no separate per-job cost** for running experiments or AutoML jobs; you only pay for underlying r

In [18]:
# Follow-up question
question4 = "Share some python code and documentation links"
print(f"\n👤 User: {question4}")

response4 = chat_sync(question4, history)
print("\033[1;31;34m")
print(f"🤖 Agent: {response4}")


👤 User: Share some python code and documentation links

🤖 Agent: Certainly! Here’s a practical example of how to do object detection using Azure Machine Learning AutoML (Python SDK v2), along with key documentation links:

---

## Example: Object Detection Training Job with Azure ML (Python SDK v2)

```python
from azure.ai.ml import MLClient, automl
from azure.identity import DefaultAzureCredential
from azure.ai.ml.automl import ObjectDetectionPrimaryMetrics

# Connect to your ML workspace
ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id="your-subscription-id",
    resource_group_name="your-resource-group",
    workspace_name="your-workspace"
)

# Create your training and validation MLTable data input (pre-registered in your workspace)
training_data_input = "azureml:object-detection-train:1"
validation_data_input = "azureml:object-detection-valid:1"

# Configure and create the AutoML job
image_object_detection_job = automl.image_object_detection(
    compute="yo

## 6. Interactive Chatbot Interface with Multi-Turn Support

The chatbot now maintains conversation context for follow-up questions.

In [19]:
EXAMPLE_QUESTIONS = [
    "What are the available object detection models?",
    "How to do OCR?",
    "How to do geocoding?",
    "What is Azure Content Understanding?",
    "How to use Azure AI Search with vector embeddings?",
    "What are the Text to Speech options in Azure?",
    "What are the recent news about Speech services?",
    "What is PTU?",
]

FOLLOWUP_SUGGESTIONS = [
    "Translate the answer to French",
    "Give some pricing informations",
    "Share some documentation links",
    "Give me a Python code example",
    "Summarize the answer",
]

In [20]:
# Global state to track current conversation ID
current_conv_state = {"id": None}


def get_dropdown_choices():
    """Get formatted choices for the conversation dropdown."""
    conversations = history_manager.get_conversation_list()
    choices = []
    for conv in conversations:
        title = conv['title'][:35] + "..." if len(
            conv['title']) > 35 else conv['title']
        label = f"[{conv['id']}] {title} ({conv['message_count']} msgs)"
        choices.append(label)
    return choices


def extract_conv_id(dropdown_value: str) -> Optional[str]:
    """Extract conversation ID from dropdown label."""
    if not dropdown_value:
        return None
    if dropdown_value.startswith("["):
        end_bracket = dropdown_value.find("]")
        if end_bracket > 0:
            return dropdown_value[1:end_bracket]
    return None


def load_conversation_from_dropdown(dropdown_value: str):
    """Load a conversation based on dropdown selection."""
    conv_id = extract_conv_id(dropdown_value)
    if not conv_id:
        return []

    current_conv_state["id"] = conv_id
    history_manager.set_current_conversation(conv_id)
    messages = history_manager.get_messages(conv_id)

    return [{"role": m["role"], "content": m["content"]} for m in messages]


def respond(message: str, chat_history: list):
    """
    Process user message with conversation context.
    """
    if not message or not message.strip():
        return "", chat_history, gr.update()

    if chat_history is None:
        chat_history = []

    # Ensure we have a current conversation
    if not current_conv_state["id"]:
        current_conv_state["id"] = history_manager.get_current_conversation_id(
        )

    # Convert chat_history to the format expected by the agent
    # This provides context for follow-up questions
    context_history = []
    for msg in chat_history:
        if isinstance(msg, dict):
            context_history.append(msg)

    # Get response from agent WITH conversation context
    try:
        response = chat_sync(message, context_history)
    except Exception as e:
        response = f"❌ Error: {str(e)}\n\nPlease check your Azure credentials."

    # Save to persistent storage
    history_manager.add_message("user", message)
    history_manager.add_message("assistant", response)

    # Add to UI history
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": response})

    # Update dropdown
    choices = get_dropdown_choices()
    current_label = None
    for c in choices:
        if c.startswith(f"[{current_conv_state['id']}]"):
            current_label = c
            break

    return "", chat_history, gr.update(choices=choices, value=current_label)


def new_conversation():
    """Create a new conversation."""
    conv_id = history_manager.create_conversation()
    current_conv_state["id"] = conv_id

    choices = get_dropdown_choices()
    new_label = None
    for c in choices:
        if c.startswith(f"[{conv_id}]"):
            new_label = c
            break

    return gr.update(choices=choices, value=new_label), []


def delete_current_conversation(dropdown_value: str):
    """Delete the selected conversation."""
    conv_id = extract_conv_id(dropdown_value)
    if conv_id:
        history_manager.delete_conversation(conv_id)

    choices = get_dropdown_choices()

    if choices:
        new_conv_id = extract_conv_id(choices[0])
        current_conv_state["id"] = new_conv_id
        if new_conv_id:
            history_manager.set_current_conversation(new_conv_id)
        messages = load_conversation_from_dropdown(choices[0])
        return gr.update(choices=choices, value=choices[0]), messages
    else:
        conv_id = history_manager.create_conversation()
        current_conv_state["id"] = conv_id
        choices = get_dropdown_choices()
        return gr.update(choices=choices,
                         value=choices[0] if choices else None), []


def export_current_conversation(dropdown_value: str):
    """Export the current conversation as text."""
    conv_id = extract_conv_id(dropdown_value)
    return history_manager.export_conversation(conv_id)

In [21]:
def build_chatbot_interface():
    """
    Build the Gradio chatbot interface with multi-turn conversation support.
    """

    # Ensure we have at least one conversation
    if not history_manager.get_conversation_list():
        history_manager.create_conversation()

    # Get initial state
    initial_choices = get_dropdown_choices()
    initial_conv_id = history_manager.get_current_conversation_id()
    current_conv_state["id"] = initial_conv_id

    initial_dropdown_value = None
    for c in initial_choices:
        if c.startswith(f"[{initial_conv_id}]"):
            initial_dropdown_value = c
            break
    if not initial_dropdown_value and initial_choices:
        initial_dropdown_value = initial_choices[0]

    initial_messages = []
    if initial_conv_id:
        msgs = history_manager.get_messages(initial_conv_id)
        initial_messages = [{
            "role": m["role"],
            "content": m["content"]
        } for m in msgs]

    with gr.Blocks(
            title="Microsoft Learn Documentation MCP Chatbot") as interface:

        # Header
        gr.Markdown("""
            # 🔍 Microsoft Learn Documentation Chatbot using MCP
            
            Ask questions about Microsoft technologies, Azure services, and development tools.
            """)

        # Info box
        gr.HTML(f"""
            <div style="background: linear-gradient(135deg, #0078d4 0%, #00bcf2 100%); 
                        padding: 1rem; border-radius: 10px; color: white; margin-bottom: 1rem;">
            📅 <b>Date:</b> {datetime.today().strftime('%d %B %Y')} | 
            🤖 <b>Model:</b> {AZURE_AI_MODEL_DEPLOYMENT_NAME or 'Not configured'} |
            📚 <b>Source:</b> Microsoft Learn |
            💾 <b>History:</b> Persistent |
            🔄 <b>Context:</b> Last {MAX_CONTEXT_MESSAGES} messages
            </div>
            """)

        # Conversation history management
        with gr.Row():
            with gr.Column(scale=3):
                conversation_dropdown = gr.Dropdown(
                    choices=initial_choices,
                    value=initial_dropdown_value,
                    label="📂 Conversation History",
                    interactive=True,
                    allow_custom_value=False,
                )
            with gr.Column(scale=1):
                new_conv_btn = gr.Button("➕ NEW", variant="primary", size="sm")
            with gr.Column(scale=1):
                delete_conv_btn = gr.Button("🗑️ DELETE",
                                            variant="secondary",
                                            size="sm")
            with gr.Column(scale=1):
                export_btn = gr.Button("📤 EXPORT",
                                       variant="secondary",
                                       size="sm")

        # Chat interface
        chatbot = gr.Chatbot(
            label="Conversation",
            height=400,
            value=initial_messages,
        )

        # Input area
        with gr.Row():
            msg = gr.Textbox(
                label="Your question",
                placeholder=
                "Ask anything about Azure technologies... or ask a follow-up question!",
                scale=4,
                lines=2,
            )
            submit_btn = gr.Button("SEND", variant="primary", scale=1)

        # Follow-up suggestions
        gr.Markdown("### 🔄 Quick Follow-ups")
        with gr.Row():
            followup_btns = []
            for suggestion in FOLLOWUP_SUGGESTIONS:
                btn = gr.Button(suggestion, size="sm", variant="secondary")
                followup_btns.append(btn)

        # Example questions
        gr.Markdown("### 💡 Examples")

        with gr.Row():
            btn1 = gr.Button(EXAMPLE_QUESTIONS[0], size="sm")
            btn2 = gr.Button(EXAMPLE_QUESTIONS[1], size="sm")
            btn3 = gr.Button(EXAMPLE_QUESTIONS[2], size="sm")
            btn4 = gr.Button(EXAMPLE_QUESTIONS[3], size="sm")

        with gr.Row():
            btn5 = gr.Button(EXAMPLE_QUESTIONS[4], size="sm")
            btn6 = gr.Button(EXAMPLE_QUESTIONS[5], size="sm")
            btn7 = gr.Button(EXAMPLE_QUESTIONS[6], size="sm")
            btn8 = gr.Button(EXAMPLE_QUESTIONS[7], size="sm")

        # Export output
        with gr.Accordion("📋 Exported Conversation", open=False):
            export_output = gr.Textbox(
                label="Conversation Export",
                lines=10,
                interactive=False,
            )

        # Wire up follow-up buttons
        for i, btn in enumerate(followup_btns):
            btn.click(lambda s=FOLLOWUP_SUGGESTIONS[i]: s, outputs=msg)

        # Wire up example buttons
        btn1.click(lambda: EXAMPLE_QUESTIONS[0], outputs=msg)
        btn2.click(lambda: EXAMPLE_QUESTIONS[1], outputs=msg)
        btn3.click(lambda: EXAMPLE_QUESTIONS[2], outputs=msg)
        btn4.click(lambda: EXAMPLE_QUESTIONS[3], outputs=msg)
        btn5.click(lambda: EXAMPLE_QUESTIONS[4], outputs=msg)
        btn6.click(lambda: EXAMPLE_QUESTIONS[5], outputs=msg)
        btn7.click(lambda: EXAMPLE_QUESTIONS[6], outputs=msg)
        btn8.click(lambda: EXAMPLE_QUESTIONS[7], outputs=msg)

        # Wire up main events
        submit_btn.click(respond, [msg, chatbot],
                         [msg, chatbot, conversation_dropdown])
        msg.submit(respond, [msg, chatbot],
                   [msg, chatbot, conversation_dropdown])

        # Conversation management events
        conversation_dropdown.change(load_conversation_from_dropdown,
                                     inputs=[conversation_dropdown],
                                     outputs=[chatbot])

        new_conv_btn.click(new_conversation,
                           outputs=[conversation_dropdown, chatbot])

        delete_conv_btn.click(delete_current_conversation,
                              inputs=[conversation_dropdown],
                              outputs=[conversation_dropdown, chatbot])

        export_btn.click(export_current_conversation,
                         inputs=[conversation_dropdown],
                         outputs=[export_output])

        # Footer
        gr.Markdown("""
            ---
            <center>
            <small>
            💻 Powered by Azure AI Agent Framework | 
            📖 Data source: <a href="https://learn.microsoft.com" target="_blank">Microsoft Learn</a> |
            💾 Conversations saved to <code>history.json</code>
            </small>
            </center>
            """)

    return interface

In [22]:
# My theme
mytheme = gr.themes.Base(
    primary_hue=gr.themes.colors.blue,
    secondary_hue=gr.themes.colors.cyan,
    neutral_hue=gr.themes.colors.slate,
).set(
    # === BACKGROUNDS ===
    body_background_fill="#f0f4f8",
    body_background_fill_dark="#1a1a2e",
    background_fill_primary="#ffffff",
    background_fill_primary_dark="#1e1e2f",
    background_fill_secondary="#f8fafc",
    background_fill_secondary_dark="#252536",

    # === BUTTONS PRIMARY ===
    button_primary_background_fill="#0078d4",
    button_primary_background_fill_hover="#106ebe",
    button_primary_background_fill_dark="#0078d4",
    button_primary_text_color="white",
    button_primary_text_color_dark="white",
    button_primary_border_color="transparent",

    # === BUTTONS SECONDARY ===
    button_secondary_background_fill="#ffffff",
    button_secondary_background_fill_hover="#e8f4fc",
    button_secondary_background_fill_dark="#2d2d44",
    button_secondary_text_color="#0078d4",
    button_secondary_text_color_dark="#00bcf2",
    button_secondary_border_color="#0078d4",

    # === INPUTS ===
    input_background_fill="#ffffff",
    input_background_fill_dark="#2d2d44",
    input_border_color="#d1d5db",
    input_border_color_hover="#0078d4",
    input_border_color_focus="#0078d4",
    input_border_color_dark="#404055",
    input_placeholder_color="#9ca3af",

    # === TEXT COLORS ===
    body_text_color="#1f2937",
    body_text_color_dark="#e5e7eb",
    body_text_color_subdued="#6b7280",
    body_text_color_subdued_dark="#9ca3af",
    block_title_text_color="#0078d4",
    block_title_text_color_dark="#00bcf2",
    block_label_text_color="#333333",
    block_label_text_color_dark="#d1d5db",

    # === BLOCKS & PANELS ===
    block_background_fill="#ffffff",
    block_background_fill_dark="#1e1e2f",
    block_border_color="#e5e7eb",
    block_border_color_dark="#374151",
    block_border_width="1px",
    block_radius="12px",
    block_shadow="0 4px 6px -1px rgba(0, 0, 0, 0.1)",
    block_shadow_dark="0 4px 6px -1px rgba(0, 0, 0, 0.3)",
    block_title_padding="12px 16px",
    block_padding="16px",

    # === BORDERS & RADIUS ===
    border_color_accent="#0078d4",
    border_color_accent_dark="#00bcf2",
    border_color_primary="#e5e7eb",
    border_color_primary_dark="#374151",

    # === SHADOWS ===
    shadow_spread="4px",
    shadow_drop="0 10px 15px -3px rgba(0, 0, 0, 0.1)",
    shadow_drop_lg="0 20px 40px -10px rgba(0, 0, 0, 0.15)",

    # === CHECKBOXES ===
    checkbox_background_color="#ffffff",
    checkbox_background_color_dark="#2d2d44",
    checkbox_background_color_selected="#0078d4",
    checkbox_border_color="#d1d5db",
    checkbox_border_color_selected="#0078d4",

    # === TABLES ===
    table_border_color="#e5e7eb",
    table_border_color_dark="#374151",
    table_even_background_fill="#f9fafb",
    table_even_background_fill_dark="#252536",
    table_odd_background_fill="#ffffff",
    table_odd_background_fill_dark="#1e1e2f",
    table_row_focus="#eff6ff",

    # === LOADER ===
    loader_color="#0078d4",
    loader_color_dark="#00bcf2",

    # === SPACING ===
    layout_gap="16px",
    section_header_text_size="18px",
    section_header_text_weight="600",
)

In [23]:
# Build and launch the webapp
print("Running the webapp...")
webapp = build_chatbot_interface()

print(f"💾 History file: history.json")
print(f"🔄 Context window: {MAX_CONTEXT_MESSAGES} messages")
print(f"{'='*60}\n")

webapp.launch(share=True, inline=True, theme=mytheme)

Running the webapp...
💾 History file: history.json
🔄 Context window: 10 messages

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c7702ff1ec7d0f9dd0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Utility Functions

In [24]:
# View all conversations
print("📚 All conversations:")

for conv in history_manager.get_conversation_list():
    print(f"\n🆔 ID: {conv['id']}")
    print(f"   📝 First question: {conv['title']}")
    print(f"   💬 Messages: {conv['message_count']}")
    print(f"   📅 Created: {conv['created_at']}")
    print(f"   🔄 Updated: {conv['updated_at']}")

📚 All conversations:

🆔 ID: e0763025
   📝 First question: How to do OCR?
   💬 Messages: 10
   📅 Created: 2026-01-16T08:47:29.006622
   🔄 Updated: 2026-01-16T08:54:22.319458


In [25]:
# Export current conversation
export_text = history_manager.export_conversation()
print(export_text)

# How to do OCR?
Created: 2026-01-16T08:47:29.006622
Last updated: 2026-01-16T08:54:22.319458

---

👤 **User:**
How to do OCR?

🤖 **Assistant:**
To perform OCR (Optical Character Recognition) on Microsoft Azure, you can use the **Azure AI Vision Read API**, which is part of the Azure AI Services. This service allows you to extract printed and handwritten text from images and documents.

### Steps to Perform OCR Using Azure

#### 1. **Create an Azure AI Vision Resource**
First, create a [Computer Vision (Vision) resource](https://portal.azure.com/) in the Azure Portal to obtain your endpoint and API key.

#### 2. **Choose Your Method**
You can do OCR via the REST API, client SDKs (e.g., Python, C#, Java), or the Azure Portal. Below is a Python SDK example.

#### 3. **Install Required Package (Python Example)**
```bash
pip install azure-ai-vision
```

#### 4. **Sample Python Code for OCR**
```python
from azure.ai.vision import VisionClient
from azure.core.credentials import AzureKeyCrede

## 8. Alternative: CLI with Multi-Turn Support

In [26]:
def cli_agent():
    """
    Run an interactive command-line chat session with multi-turn support.
    """
    print("\n" + "=" * 60)
    print("🔍 Microsoft Learn Documentation Assistant")
    print("=" * 60)
    print("Commands:")
    print("  /new     - Start a new conversation")
    print("  /list    - List all conversations")
    print("  /load ID - Load a specific conversation")
    print("  /export  - Export current conversation")
    print("  /quit    - Exit the assistant")
    print("")
    print("💡 Tip: Ask follow-up questions like 'translate to French'")
    print("=" * 60 + "\n")

    history_manager.get_current_conversation_id()
    session_history = []  # In-memory history for context

    while True:
        try:
            user_input = input("\n📝 You: ").strip()

            if not user_input:
                continue

            # Handle commands
            if user_input.startswith("/"):
                cmd_parts = user_input.split()
                cmd = cmd_parts[0].lower()

                if cmd in ["/quit", "/exit", "/q"]:
                    print("\n👋 Goodbye!")
                    break
                elif cmd == "/new":
                    conv_id = history_manager.create_conversation()
                    session_history = []  # Reset context
                    print(f"✅ Created new conversation: {conv_id}")
                elif cmd == "/list":
                    convs = history_manager.get_conversation_list()
                    print("\n📚 Conversations:")
                    for c in convs:
                        marker = "→" if c[
                            "id"] == history_manager.get_current_conversation_id(
                            ) else " "
                        print(
                            f"  {marker} [{c['id']}] {c['title']} ({c['message_count']} msgs)"
                        )
                elif cmd == "/load" and len(cmd_parts) > 1:
                    conv_id = cmd_parts[1]
                    if history_manager.set_current_conversation(conv_id):
                        # Load history for context
                        session_history = history_manager.get_messages(conv_id)
                        print(
                            f"✅ Loaded conversation: {conv_id} ({len(session_history)} messages)"
                        )
                    else:
                        print(f"❌ Conversation not found: {conv_id}")
                elif cmd == "/export":
                    print(history_manager.export_conversation())
                else:
                    print("❓ Unknown command.")
                continue

            # Regular message with context
            print("\n⏳ Searching documentation...")
            response = chat_sync(user_input, session_history)

            # Save to persistent storage
            history_manager.add_message("user", user_input)
            history_manager.add_message("assistant", response)

            # Update session history for context
            session_history.append({"role": "user", "content": user_input})
            session_history.append({"role": "assistant", "content": response})

            print(f"\n🤖 Assistant:\n{response}")

        except KeyboardInterrupt:
            print("\n\n👋 Session interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")

In [27]:
cli_agent()


🔍 Microsoft Learn Documentation Assistant
Commands:
  /new     - Start a new conversation
  /list    - List all conversations
  /load ID - Load a specific conversation
  /export  - Export current conversation
  /quit    - Exit the assistant

💡 Tip: Ask follow-up questions like 'translate to French'




📝 You:  hello



⏳ Searching documentation...

🤖 Assistant:
Hello! How can I assist you with Microsoft Azure or any related technology today? If you have questions or need guidance, feel free to ask!



📝 You:  what is azure fabric?



⏳ Searching documentation...

🤖 Assistant:
Azure Fabric typically refers to Azure Service Fabric, which is a distributed systems platform used to build scalable, reliable, and easily managed applications for the cloud and on-premises environments.

### Key Features of Azure Service Fabric:
- **Microservices Platform:** Supports building and managing microservices and container-based applications.
- **Scalability and Reliability:** Provides automatic scaling, rolling upgrades, and self-healing.
- **Orchestration:** Manages the placement, deployment, and resiliency of services.
- **Programming Models:** Supports stateless and stateful services, containers, and various programming languages like .NET and Java.
- **DevOps Integration:** Supports DevOps processes with rolling upgrades, diagnostics, and monitoring.

### Use Cases
- Large-scale systems (e.g., Azure SQL Database, Azure Cosmos DB run on Service Fabric)
- Real-time data processing
- High-availability enterprise applications

Fo


📝 You:  /quit



👋 Goodbye!


### History.json display

In [28]:
!ls history.json -lh

-rwxrwxrwx 1 root root 16K Jan 16 08:55 history.json


In [29]:
with open('history.json', 'r') as file:
    data = json.load(file)

print(json.dumps(data, indent=4))

{
    "conversations": {
        "e0763025": {
            "id": "e0763025",
            "title": "How to do OCR?",
            "created_at": "2026-01-16T08:47:29.006622",
            "updated_at": "2026-01-16T08:55:21.990759",
            "messages": [
                {
                    "role": "user",
                    "content": "How to do OCR?",
                    "timestamp": "2026-01-16T08:47:54.384912"
                },
                {
                    "role": "assistant",
                    "content": "To perform OCR (Optical Character Recognition) on Microsoft Azure, you can use the **Azure AI Vision Read API**, which is part of the Azure AI Services. This service allows you to extract printed and handwritten text from images and documents.\n\n### Steps to Perform OCR Using Azure\n\n#### 1. **Create an Azure AI Vision Resource**\nFirst, create a [Computer Vision (Vision) resource](https://portal.azure.com/) in the Azure Portal to obtain your endpoint and API key.\